In [2]:
import pandas as pd
from randomforest import RandomForestModel
from dataloader import DataLoader
from backtest import RandomForestStrategy
import yfinance as yf
import backtrader as bt
import warnings
import numpy as np

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

dl = DataLoader()
rf = RandomForestModel(dl)


In [3]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

def correlation_matrix():
    df = dl.get_data()
    
    numeric_df = df.select_dtypes(include=[np.number])
    corr_matrix = numeric_df.corr()    
    
    plt.figure(figsize=(20, 15))  
    sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', linewidths=0.5)
    plt.title('Correlation Matrix')
    plt.show()

correlation_matrix()


    

In [4]:
rf.run_model()
rf.get_overall_results()


In [6]:
import matplotlib.pyplot as plt
import numpy as np
%matplotlib inline

etf_data = yf.Ticker("IHYG.L").history(start=dl.start, end=dl.end)
etf_data.index = pd.to_datetime(etf_data.index).tz_localize(None)
predictions = rf.get_predictions()

etf_data["Signal"] = etf_data.index.map(lambda k: predictions.get(k.date()))
print(etf_data["Signal"].value_counts())

etf_data["Next_Open"] = etf_data["Open"].shift(-1)
etf_data["Returns"] = etf_data["Signal"] * ((etf_data["Next_Open"] / etf_data["Close"]) - 1)
etf_data["Cumulative_Returns"] = (1 + etf_data["Returns"]).cumprod()
etf_data["Cumulative_Returns"].plot(title="Backtest: Cumulative Returns")
plt.show()

print(f"Cumulative Returns: {etf_data["Cumulative_Returns"].iloc[-2]:.4f}")
print(f"Sharpe Ratio: {etf_data["Returns"].mean() / etf_data["Returns"].std() * np.sqrt(252):.4f}")
print(f"Total Trades: {np.sum(etf_data["Signal"].diff() != 0)}")
print(f"Winning Trades: {np.sum(etf_data["Returns"] > 0)}")
print(f"Losing Trades: {np.sum(etf_data["Returns"] < 0)}")


In [6]:
def run_backtest():
    df = rf.df.copy()

    volume_series = yf.Ticker("IHYG.L").history(start=dl.start, end=dl.end)["Volume"]
    volume_series.index = volume_series.index.tz_localize(None)
    df["Volume"] = volume_series.reindex(df.index).fillna(0)

    vix_series = yf.Ticker("^VIX").history(start=dl.start, end=dl.end)["Close"]
    vix_series.index = vix_series.index.tz_localize(None)
    df["VIX"] = vix_series.reindex(df.index).fillna(0)

    predictions = {key.date(): value for key, value in rf.get_predictions().items()}
    volatility = {key.date(): value for key, value in zip(df.index, df["VIX"])}

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    data_feed = bt.feeds.PandasData(dataname=df)

    cerebro = bt.Cerebro()
    cerebro.adddata(data_feed)
    cerebro.broker.setcash(1_000_000)
    cerebro.broker.set_coo(True)
    cerebro.broker.set_coc(True)
    cerebro.addstrategy(RandomForestStrategy, predictions=predictions, volatility=volatility)

    cerebro.run()
    cerebro.plot()

run_backtest()
